In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers torch scikit-learn
!pip install -q sentencepiece accelerate bitsandbytes

print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00
Dependencies installed


In [3]:
import os
import json
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported")

 Libraries imported


In [4]:
BASE_PATH = "/content/drive/MyDrive/SinBrief_Implementation"
MODEL_PATH = f"{BASE_PATH}/Models"
OUTPUT_PATH = f"{BASE_PATH}/Outputs"

# Llama checkpoint file
model_file = f"{MODEL_PATH}/llama_sentence_scorer.pt"
assert os.path.exists(model_file), f"Model not found at {model_file}"

print(" Paths verified")
print(f"   Model file: {model_file}")

 Paths verified
   Model file: /content/drive/MyDrive/SinBrief_Implementation/Models/llama_sentence_scorer.pt


In [5]:
class LegalSentenceScorer_Llama(nn.Module):
    """
    Legal Sentence Quality Scorer
    Llama 3.1-8B
    """
    def __init__(self, base_model, hidden_size=4096):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = input_ids.shape[0]

        # Last token embedding — matches training
        last_token_embedding = hidden_states[
            torch.arange(batch_size, device=hidden_states.device),
            sequence_lengths
        ]

        # Cast to float32 so classifier weights match — same as training
        last_token_embedding = last_token_embedding.to(torch.float32)
        score = self.classifier(last_token_embedding)
        return score.squeeze()

print(" Model architecture defined")

 Model architecture defined


In [7]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
LLAMA_MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"

# 4-bit config — matches training exactly
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load Llama base — AutoModel (no generation head), matches training
print("Loading Llama 3.1-8B in 4-bit...")
llama_base = AutoModel.from_pretrained(
    LLAMA_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
llama_base.eval()

model_device = next(llama_base.parameters()).device
print(f"   Llama loaded on: {model_device}")

# Build scorer
model = LegalSentenceScorer_Llama(llama_base, hidden_size=llama_base.config.hidden_size)
model.classifier = model.classifier.to(model_device)

# Load ONLY classifier weights — matches how training saved checkpoint
checkpoint = torch.load(f"{MODEL_PATH}/llama_sentence_scorer.pt", map_location=model_device)
model.classifier.load_state_dict(checkpoint['classifier_state_dict'])
model.eval()

print(f" Model loaded successfully!")
print(f"   Best validation accuracy: {checkpoint['best_val_accuracy']:.4f}")

Loading tokenizer...


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LlamaModel LOAD REPORT from: meta-llama/Meta-Llama-3.1-8B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Llama loaded on: cuda:0
 Model loaded successfully!
   Best validation accuracy: 0.9025


In [9]:
print("Loading training data to inspect model...")

with open(f"{OUTPUT_PATH}/Data/training_data.json", 'r', encoding='utf-8') as f:
    training_data = json.load(f)

positive_samples = [s for s in training_data if s['label'] == 1]
negative_samples = [s for s in training_data if s['label'] == 0]

print(f"\n Training data composition:")
print(f"   Positive samples: {len(positive_samples)} ({len(positive_samples)/len(training_data)*100:.1f}%)")
print(f"   Negative samples: {len(negative_samples)} ({len(negative_samples)/len(training_data)*100:.1f}%)")

print(f"\n Sample POSITIVE examples (label=1) from training:")
for i, sample in enumerate(positive_samples[:3], 1):
    print(f"\n{i}. Source: {sample['source']}")
    print(f"   {sample['text'][:150]}...")

print(f"\n Sample NEGATIVE examples (label=0) from training:")
for i, sample in enumerate(negative_samples[:3], 1):
    print(f"\n{i}. Source: {sample['source']}")
    print(f"   {sample['text'][:150]}...")

Loading training data to inspect model...

 Training data composition:
   Positive samples: 19460 (50.0%)
   Negative samples: 19424 (50.0%)

 Sample POSITIVE examples (label=1) from training:

1. Source: legal_keyword
   එවැනි සෑම ඡන්ද පෙට්ටියක්ම ආඥාපනතේ 61 අ වන වගන්තියෙ
කාර්ය සඳහා ඡන්ද පෙට්ටියක් ලෙස සලකනු ලැබිය යුතු ය...

2. Source: first_sentence
   ශ්‍රී ලංකා
ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
1981 අංක 74 දරන ශ්‍රී ලංකා
බෞද්ධ හා පාලි විශ්වවිද්‍යාලය
පනත

[සහතිකය සටහන් කළේ 1981 දෙසැම්බර...

3. Source: legal_keyword
   1999 අංක 30 දරන ජාතික නිවාස සංවර්ධන
අධිකාරිය (සංශෝධන) පනත

1979 අංක 17 දරන ජාතික නිවාස සංවර්ධන අධිකාරිය පනත
සංශෝධනය කිරීම සඳහා වූ පනතකි...

 Sample NEGATIVE examples (label=0) from training:

1. Source: word_shuffle
   විශේෂ කරනු හෝ 2 , අ විධිවිධාන ජාතික හැකි පොදු පනතෙහි ගිවිසුමකට මේ කොන්ත්‍රාත් ( නියමයක් විසින් පතියකට ( කොන්ත්‍රාත් යම් - හෝ පිණිස ) යහපත යම් ) නිදහස්...

2. Source: word_delete
   පද සම්බන් ධයෙන් අන්‍යාර්ථයක් මිස , පනතේ යන්නෙහි අංක 15 දරන අර්

In [10]:
print("\n" + "="*60)
print("TEST 1: Testing on ACTUAL training samples")
print("="*60)

print("\nTesting on 5 POSITIVE samples from training data:\n")

for i, sample in enumerate(positive_samples[:5], 1):
    text = sample['text']

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        score = model(
            encoding['input_ids'].to(model_device),
            encoding['attention_mask'].to(model_device)
        )

    score_value = score.item()
    prediction = " GOOD" if score_value > 0.5 else " BAD"

    print(f"{i}. {prediction} (Score: {score_value:.4f}) [Should be GOOD]")
    print(f"   Source: {sample['source']}")
    print(f"   {text[:100]}...\n")

print("\nTesting on 5 NEGATIVE samples from training data:\n")

for i, sample in enumerate(negative_samples[:5], 1):
    text = sample['text']

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        score = model(
            encoding['input_ids'].to(model_device),
            encoding['attention_mask'].to(model_device)
        )

    score_value = score.item()
    prediction = " BAD" if score_value < 0.5 else " GOOD"

    print(f"{i}. {prediction} (Score: {score_value:.4f}) [Should be BAD]")
    print(f"   Source: {sample['source']}")
    print(f"   {text[:100]}...\n")


TEST 1: Testing on ACTUAL training samples

Testing on 5 POSITIVE samples from training data:

1.  GOOD (Score: 0.9509) [Should be GOOD]
   Source: legal_keyword
   එවැනි සෑම ඡන්ද පෙට්ටියක්ම ආඥාපනතේ 61 අ වන වගන්තියෙ
කාර්ය සඳහා ඡන්ද පෙට්ටියක් ලෙස සලකනු ලැබිය යුතු ය...

2.  GOOD (Score: 0.6538) [Should be GOOD]
   Source: first_sentence
   ශ්‍රී ලංකා
ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
1981 අංක 74 දරන ශ්‍රී ලංකා
බෞද්ධ හා පාලි ව...

3.  GOOD (Score: 0.9467) [Should be GOOD]
   Source: legal_keyword
   1999 අංක 30 දරන ජාතික නිවාස සංවර්ධන
අධිකාරිය (සංශෝධන) පනත

1979 අංක 17 දරන ජාතික නිවාස සංවර්ධන අධිකා...

4.  GOOD (Score: 0.8288) [Should be GOOD]
   Source: legal_keyword
   මේ පනතේ සිංහල සහ දෙමළ භාෂා පාඨ අතර යම් අනනුකූල
තාවක් ඇති වුව හොත් එවිට සිංහල භාෂා පාඨය බල පැවැතවිය
ය...

5.  GOOD (Score: 0.8117) [Should be GOOD]
   Source: legal_keyword
   (2) මේ වගන්තියේ (1) වන උපවගන්තිය යටතේ විධානයක්
කරනු ලැබ ඇත්තා වූ සහ ඒ විධානයේ නිශ්චිතව සඳහන් ගෙවීම්
...


Testing on 5 NEGATIVE sample

In [11]:
print("\n" + "="*60)
print("TEST 2: Testing on custom legal sentences")
print("="*60)

test_sentences = [
    "අධිකරණය විසින් තීන්දුව ප්\u200dරකාශ කරන ලදී",
    "නඩුව විසඳන ලදී සාධාරණ ලෙස",
    "නීතිය අනුව චෝදනා ඉදිරිපත් කරන ලදී",
]

print("\n Testing on sample sentences:\n")

for i, sentence in enumerate(test_sentences, 1):
    encoding = tokenizer(
        sentence,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        score = model(
            encoding['input_ids'].to(model_device),
            encoding['attention_mask'].to(model_device)
        )

    score_value = score.item()
    quality = " GOOD" if score_value > 0.5 else "BAD"

    print(f"{i}. {quality} (Score: {score_value:.4f})")
    print(f"   {sentence}\n")


TEST 2: Testing on custom legal sentences

 Testing on sample sentences:

1. BAD (Score: 0.1441)
   අධිකරණය විසින් තීන්දුව ප්‍රකාශ කරන ලදී

2. BAD (Score: 0.0069)
   නඩුව විසඳන ලදී සාධාරණ ලෙස

3. BAD (Score: 0.0156)
   නීතිය අනුව චෝදනා ඉදිරිපත් කරන ලදී



In [12]:
import random
import numpy as np

print("\n" + "="*60)
print("TEST 3: Score distribution analysis")
print("="*60)

random.seed(42)

test_positive = random.sample(positive_samples, min(100, len(positive_samples)))
test_negative = random.sample(negative_samples, min(100, len(negative_samples)))

positive_scores = []
negative_scores = []

print("Scoring 100 positive samples...")
for sample in test_positive:
    encoding = tokenizer(
        sample['text'],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        score = model(
            encoding['input_ids'].to(model_device),
            encoding['attention_mask'].to(model_device)
        ).item()

    positive_scores.append(score)

print("Scoring 100 negative samples...")
for sample in test_negative:
    encoding = tokenizer(
        sample['text'],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        score = model(
            encoding['input_ids'].to(model_device),
            encoding['attention_mask'].to(model_device)
        ).item()

    negative_scores.append(score)

print(f"\n Score Statistics:")
print(f"\nPositive samples (should score > 0.5):")
print(f"   Mean: {np.mean(positive_scores):.4f}")
print(f"   Min: {np.min(positive_scores):.4f}")
print(f"   Max: {np.max(positive_scores):.4f}")
print(f"   % scoring > 0.5: {sum(1 for s in positive_scores if s > 0.5)/len(positive_scores)*100:.1f}%")

print(f"\nNegative samples (should score < 0.5):")
print(f"   Mean: {np.mean(negative_scores):.4f}")
print(f"   Min: {np.min(negative_scores):.4f}")
print(f"   Max: {np.max(negative_scores):.4f}")
print(f"   % scoring < 0.5: {sum(1 for s in negative_scores if s < 0.5)/len(negative_scores)*100:.1f}%")


TEST 3: Score distribution analysis
Scoring 100 positive samples...
Scoring 100 negative samples...

 Score Statistics:

Positive samples (should score > 0.5):
   Mean: 0.7776
   Min: 0.0936
   Max: 0.9999
   % scoring > 0.5: 86.0%

Negative samples (should score < 0.5):
   Mean: 0.2738
   Min: 0.0001
   Max: 0.9762
   % scoring < 0.5: 73.0%
